# Local Inference{background-color="black"}

*Running open-weight LLMs on your own hardware: llama.cpp, Ollama, vLLM*

::: {.fragment}
**Lesson 16 roadmap (3h)**

| Block | Topic | Time |
|---|---|---|
| 1 | Why local, anatomy of LLM inference, memory math, quantization, GGUF | 40' |
| 2 | **llama.cpp**: the engine (hands-on) | 35' |
| 3 | **Ollama**: the developer experience (hands-on) | 30' |
| 4 | **vLLM**: serving at scale (demo / theory) | 25' |
| 5 | The OpenAI-compatible API, benchmarking | 15' |
| 6 | **Spark as a client**: batched, concurrent enrichment | 30' |
| | Wrap-up, project tie-in | 5' |

Code: `tap2027/inference/`
:::

## From the previous lesson

```{mermaid}
flowchart LR
  subgraph Spark executor
    A[Spark NLP<br/>ONNX in JVM]
    B[Pandas UDF<br/>HF model in Python]
  end
  subgraph Inference server
    C[llama.cpp / Ollama / vLLM<br/>GGUF / safetensors]
  end
  B -. "LLM? too big" .-> C
  D[Spark mapInPandas] -- "HTTP /v1/chat/completions<br/>batched + concurrent" --> C
```

::: {.fragment}
ONNX taught us that a model can be **detached** from its training framework. Today we detach it **from the data pipeline** too: the model becomes a **service**.
:::

## Why local inference?

:::: {.columns}

::: {.fragment .column width="50%"}
**Pros**

- **Privacy / GDPR**: data (e.g. students' messages) never leaves your infrastructure
- **Cost**: no per-token billing; predictable for batch enrichment of millions of rows
- **Latency & offline**: no internet round trip, works air-gapped
- **Control & reproducibility**: pinned weights (hash), no silent model updates
- **Customization**: fine-tuned / LoRA models, custom sampling, logits access
:::

::: {.fragment .column width="50%"}
**Cons**

- Open models **≤ 30B** are good, but frontier cloud models are still better at hard reasoning
- You own the **ops**: GPUs, drivers, scaling, monitoring, security patches
- Hardware cost and utilization (an idle GPU is expensive)
- Licenses vary: Apache 2.0 / MIT vs "open weights with conditions" (Llama, Gemma)
:::
::::

::: {.fragment}
Typical pattern: **local for development and sensitive/bulk workloads**, cloud API where quality matters most — behind the *same* OpenAI-compatible interface.
:::

## Anatomy of LLM inference

:::: {.columns}

::: {.fragment .column width="50%"}
1. **Tokenize** the prompt (text → token ids)
2. **Prefill**: process all prompt tokens *in parallel* → **compute-bound**
3. **Decode**: generate **one token at a time**, each step reads *all* weights → **memory-bandwidth-bound**
4. **KV cache**: keys/values of past tokens are stored to avoid recomputation; grows with context × concurrent requests
5. **Sampling**: temperature, top-p, top-k, seed
:::

::: {.fragment .column width="50%"}
**Metrics you will measure**

| Metric | Meaning |
|---|---|
| **TTFT** | time to first token (≈ prefill) |
| **TPOT / ITL** | time per output token (decode) |
| **tokens/s** (per request) | user-perceived speed |
| **throughput** (total tok/s) | server capacity with N concurrent requests |

Latency and throughput **trade off**: batching more requests raises throughput but slows each one.
:::
::::

## Memory math: will it fit?

**Weights** ≈ parameters × bytes per parameter

| Model size | FP16 (2 B) | Q8_0 (~1.06 B) | Q4_K_M (~0.6 B) |
|---|---|---|---|
| 0.5-1B | 1-2 GB | 0.5-1 GB | 0.4-0.7 GB |
| 3-4B | 6-8 GB | 3.2-4.2 GB | 2-2.5 GB |
| 7-8B | 14-16 GB | 7.5-8.5 GB | 4.4-4.9 GB |
| 14B | 28 GB | 15 GB | 9 GB |
| 32B | 64 GB | 34 GB | 19 GB |
| 70B | 140 GB | 75 GB | 42 GB |

::: {.fragment}
**KV cache** per token = 2 × layers × kv_heads × head_dim × bytes

Llama-3.1-8B (32 layers, 8 KV heads, head_dim 128, FP16): 2·32·8·128·2 = **128 KB/token** → an 8k context ≈ **1 GB**, × each concurrent request.
:::

::: {.fragment}
**Decode speed rule of thumb**: tokens/s ≈ memory bandwidth ÷ model size

laptop DDR5 (~70 GB/s) with 4.5 GB → ~15 tok/s · Apple M-series (120-800 GB/s) → 25-150 tok/s · RTX 4090 (1 TB/s) → ~200 tok/s
:::

## Quantization and GGUF

:::: {.columns}

::: {.fragment .column width="50%"}
**Quantization** stores weights with fewer bits (16 → 8 → 4 bits)

- smaller files, less RAM, **faster decode** (fewer bytes to read)
- small quality loss; **Q4_K_M** is the usual sweet spot, Q5/Q6 if you have room, Q8 ≈ lossless
- "K-quants" quantize in blocks with per-block scales; *imatrix* quants use calibration data
- Other formats: **AWQ, GPTQ, FP8** (GPU, vLLM), **MLX** (Apple)
:::

::: {.fragment .column width="50%"}
**GGUF** = the file format of llama.cpp

- **one file**: tensors + tokenizer + chat template + metadata
- memory-mapped: loads in seconds
- naming: `Qwen2.5-7B-Instruct-Q4_K_M.gguf`
- thousands of ready GGUFs on the Hub ([filter: GGUF](https://huggingface.co/models?library=gguf)); the Hub shows a GGUF viewer and "Use this model → llama.cpp / Ollama"
- convert yourself: `convert_hf_to_gguf.py` + `llama-quantize`
:::
::::

### Choosing an open model

| Family | Sizes (examples) | License | Notes |
|---|---|---|---|
| Qwen (Alibaba) | 0.5B → 235B (MoE) | Apache 2.0 | strong multilingual, good Italian, tool calling |
| Gemma (Google) | 1B → 27B | Gemma terms | good quality/size, vision variants |
| Llama (Meta) | 1B → 70B+ | Llama community license | huge ecosystem |
| Mistral | 7B → 24B (Small) | Apache 2.0 (most) | EU company, efficient |
| Phi (Microsoft) | 3.8B-14B | MIT | small, reasoning-oriented |
| gpt-oss (OpenAI) | 20B, 120B (MoE) | Apache 2.0 | open-weight reasoning models |

::: {.fragment}
For the demos we use a **tiny** model that runs on any laptop CPU: `Qwen2.5-0.5B-Instruct` (Q4_K_M, ~400 MB). For real tasks try 3-8B. Check the [Open LLM leaderboards](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) and, above all, **evaluate on your own data**.
:::

# llama.cpp{background-color="black"}

*LLM inference in C/C++ — the engine behind most local AI*

## llama.cpp in a nutshell

:::: {.columns}

::: {.fragment .column width="50%"}
- Started by Georgi Gerganov (2023), now [ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp), MIT license
- Built on **ggml**, a tensor library with many backends: **CPU (AVX/NEON), Metal, CUDA, ROCm, Vulkan, SYCL**
- Runs from Raspberry Pi to multi-GPU servers; CPU+GPU **offloading** (`-ngl`)
- Embedded by Ollama (historically), LM Studio, Spark NLP `AutoGGUFModel`, many apps
:::

::: {.fragment .column width="50%"}
**Main tools** — recent builds ship one `llama` binary with subcommands (the old `llama-*` names still exist)

| Command | Purpose |
|---|---|
| `llama cli` | interactive chat in the terminal |
| `llama serve` (`llama-server`) | HTTP server, **OpenAI-compatible**, web UI |
| `llama bench` | performance benchmarks |
| `llama quantize` | FP16 GGUF → Q4/Q5/Q8 |
| `llama download` / `fit-params` | fetch from the Hub / compute what fits in memory |
| `convert_hf_to_gguf.py` | HF safetensors → GGUF |
:::
::::

## Hands-on: `llama-server` in Docker

```bash
cd tap2027/inference
docker compose up llamacpp
```

`docker-compose.yml` (excerpt)
```yaml
  llamacpp:
    image: ghcr.io/ggml-org/llama.cpp:server-b12429   # CPU build; server-cuda / server-vulkan for GPUs
    ports: ["8080:8080"]
    volumes: ["hf-cache:/root/.cache/huggingface"]
    command: >
      -hf Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q4_K_M
      --host 0.0.0.0 --port 8080
      -c 8192            # total context (shared by the slots)
      --parallel 4       # 4 concurrent slots -> continuous batching
      -t 4               # CPU threads
      --alias qwen-0.5b
```

::: {.fragment}
- `-hf repo:quant` downloads the GGUF from the Hub (cached in the volume)
- open **http://localhost:8080** → built-in chat web UI
- `GET /health`, `GET /v1/models`, `POST /v1/chat/completions`, `POST /v1/embeddings`, `GET /metrics` (with `--metrics`)
:::

### Talk to it

In [ ]:
%%bash
curl -s http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
        "model": "qwen-0.5b",
        "messages": [
          {"role": "system", "content": "You are a concise teaching assistant."},
          {"role": "user",   "content": "Explain Apache Kafka in one sentence."}
        ],
        "temperature": 0.2,
        "max_tokens": 64
      }' | python3 -m json.tool

::: {.fragment}
Look at the `usage` and `timings` fields: prompt tokens, predicted tokens, **prompt_per_second** (prefill) and **predicted_per_second** (decode).
:::

::: {.fragment}
**Benchmark the hardware** (the model is already in the container's HF cache)
```bash
docker compose exec llamacpp sh -c '/app/llama bench -t 4 -p 512 -n 128 \
  -m /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/*/*.gguf'
```
```text
| model                  |       size | backend | threads |  test |          t/s |
| qwen2 1B Q4_K - Medium | 462.96 MiB | CPU     |       4 | pp512 | 293.97 ± 4.35|
| qwen2 1B Q4_K - Medium | 462.96 MiB | CPU     |       4 | tg128 |  65.31 ± 2.17|
```
`pp512` = prefill tokens/s, `tg128` = generation tokens/s (4 threads, laptop CPU). 65 tok/s × 0.46 GB ≈ 30 GB/s effective bandwidth: compare with the rule of thumb!
:::

## Important `llama-server` options

| Option | Meaning |
|---|---|
| `-m file.gguf` / `-hf repo[:quant]` | model from disk / from the Hub |
| `-c N` | context size (total, divided among slots) |
| `--parallel N` (`-np`) | concurrent requests (slots), continuous batching |
| `-ngl N` | layers offloaded to GPU (`99` = all) |
| `-t N` | CPU threads (≈ physical cores) |
| `--jinja` | use the model's chat template (needed for tool calling) |
| `--embeddings` | serve `/v1/embeddings` with an embedding model |
| `--api-key KEY` | require `Authorization: Bearer KEY` |
| `-fa` | flash attention (faster, less memory) |
| `--cache-type-k q8_0` | quantize the KV cache to fit longer contexts |

Every option also exists as env var `LLAMA_ARG_*` (handy in Compose/Kubernetes).

# Ollama{background-color="black"}

*"Docker for LLMs": pull, run, serve*

## Ollama in a nutshell

:::: {.columns}

::: {.fragment .column width="50%"}
- Go application wrapping a GGML-based engine (llama.cpp-derived + its own engine for new architectures)
- **Model library** with curated tags: `ollama pull qwen2.5:0.5b`
- Manages **model lifecycle**: loads on first request, unloads after `keep_alive` (default 5 min), multiple models, GPU detection
- Native API (`/api/chat`, `/api/generate`, `/api/embed`) **and** OpenAI-compatible `/v1/...`
- Runs on macOS, Windows, Linux, Docker; MIT license
:::

::: {.fragment .column width="50%"}
| Docker | Ollama |
|---|---|
| `docker pull` | `ollama pull qwen2.5:0.5b` |
| `docker run -it` | `ollama run qwen2.5:0.5b` |
| `docker images` | `ollama list` |
| `docker ps` | `ollama ps` |
| `docker rm` | `ollama rm` |
| `Dockerfile` | `Modelfile` |
| Docker Hub | ollama.com/library (+ any GGUF on HF: `ollama run hf.co/user/repo:Q4_K_M`) |
:::
::::

## Hands-on: Ollama in Docker

```bash
docker compose up -d ollama
docker compose exec ollama ollama pull qwen2.5:0.5b
docker compose exec ollama ollama pull nomic-embed-text      # embeddings for the RAG lesson
docker compose exec ollama ollama run qwen2.5:0.5b "What is a Kafka consumer group?"
docker compose exec ollama ollama ps                          # what is loaded, CPU/GPU split
```

`docker-compose.yml` (excerpt)
```yaml
  ollama:
    image: ollama/ollama:0.40.16
    ports: ["11434:11434"]
    volumes: ["ollama:/root/.ollama"]
    environment:
      OLLAMA_NUM_PARALLEL: 4          # concurrent requests per model
      OLLAMA_MAX_LOADED_MODELS: 2
      OLLAMA_KEEP_ALIVE: 30m
      OLLAMA_CONTEXT_LENGTH: 8192
```

### Native API and structured output

In [ ]:
import requests, json

schema = {"type": "object",
          "properties": {"sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
                         "topic": {"type": "string"}},
          "required": ["sentiment", "topic"]}

r = requests.post("http://localhost:11434/api/chat", json={
    "model": "qwen2.5:0.5b",
    "messages": [{"role": "user", "content": "Classify: 'The Spark lab today was awesome, finally Kafka works!'"}],
    "format": schema,                    # constrained decoding: output MUST match the JSON schema
    "stream": False,
    "options": {"temperature": 0, "seed": 42},
})
print(json.loads(r.json()["message"]["content"]))

::: {.fragment}
**Structured output** (JSON schema → grammar-constrained sampling) is what makes LLMs usable **inside a data pipeline**: every row returns parseable, typed fields. llama.cpp (`response_format` / `json_schema`) and vLLM (`guided_json` / `response_format`) support it too.
:::

## Modelfile: package a model with its configuration

```dockerfile
# inference/Modelfile
FROM qwen2.5:0.5b
PARAMETER temperature 0.1
PARAMETER num_ctx 4096
SYSTEM """You are the TAP course assistant at UniCT.
Answer briefly, in the language of the question. If you do not know, say so."""
```

```bash
docker compose exec ollama ollama create tap-assistant -f /work/Modelfile
docker compose exec ollama ollama run tap-assistant "Quando è l'esame di TAP?"
```

::: {.fragment}
Notice the honest answer (or the hallucination!) → that is why we need **RAG** with approved sources (next lesson).
:::

# vLLM{background-color="black"}

*High-throughput serving engine for production*

## vLLM in a nutshell

:::: {.columns}

::: {.fragment .column width="50%"}
- Born at UC Berkeley (2023), now a PyTorch Foundation project, Apache 2.0
- **PagedAttention**: KV cache stored in fixed-size **pages** (like OS virtual memory) → almost no fragmentation, many more concurrent requests
- **Continuous batching**: requests join/leave the running batch at every decode step
- **Prefix caching**: shared prompt prefixes (system prompt, RAG context) computed once
- Chunked prefill, speculative decoding, **tensor/pipeline parallelism** across GPUs/nodes
- Quantization: FP8, AWQ, GPTQ, bitsandbytes, (GGUF experimental)
:::

::: {.fragment .column width="50%"}
```bash
# NVIDIA GPU required (CPU/ROCm/TPU builds exist but are not the main target)
docker run --gpus all -p 8000:8000 \
  -v hf-cache:/root/.cache/huggingface \
  vllm/vllm-openai:v0.29.0 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.85 \
  --max-num-seqs 64
```

or `docker compose --profile gpu up vllm`

- loads **safetensors** directly from the Hub (no GGUF conversion)
- `/v1/chat/completions`, `/v1/completions`, `/v1/embeddings`, `/metrics` (Prometheus)
- pre-allocates `gpu-memory-utilization` of VRAM for weights + KV cache pages
:::
::::

## Why throughput explodes with vLLM

```{mermaid}
gantt
  dateFormat X
  axisFormat %s
  section Static batching
  req A (long)  :0, 8
  req B (short) :0, 3
  idle slot     :crit, 3, 8
  req C waits   :8, 11
  section Continuous batching
  req A (long)  :0, 8
  req B (short) :0, 3
  req C joins   :3, 6
  req D joins   :6, 9
```

::: {.fragment}
With 64 concurrent requests a single GPU can deliver **thousands of tokens/s** in total, while each user still sees 30-80 tok/s. This is what you want when **Spark sends many requests at once**.
:::

::: {.fragment}
Similar engines: **SGLang** (RadixAttention, strong for structured/agentic workloads), **TensorRT-LLM** / NVIDIA Dynamo, Hugging Face **TGI**. On Kubernetes: KServe, llm-d, vLLM production stack.
:::

## llama.cpp vs Ollama vs vLLM

| | llama.cpp | Ollama | vLLM |
|---|---|---|---|
| Role | engine + lightweight server | developer UX on top of an engine | production serving engine |
| Hardware | CPU, Apple, any GPU | CPU, Apple, NVIDIA/AMD | NVIDIA (main), AMD, TPU, ... |
| Model format | GGUF | GGUF (library / HF) | safetensors (HF), FP8/AWQ/GPTQ |
| Concurrency | slots (`--parallel`) | `OLLAMA_NUM_PARALLEL` | PagedAttention + continuous batching, hundreds of seqs |
| Multi-GPU / multi-node | basic split | basic | tensor + pipeline parallel |
| Setup effort | low | **lowest** | medium (GPU, drivers) |
| Best for | edge, laptops, embedding in apps | dev laptops, prototypes, small teams | shared GPU servers, high QPS, batch enrichment |

::: {.fragment}
**Rule of thumb**: prototype with **Ollama**, ship to edge/CPU with **llama.cpp**, serve many users or big batch jobs on GPUs with **vLLM**. The client code does not change.
:::

## One contract: the OpenAI-compatible API

The same client works against all three servers (and cloud providers): only `base_url` changes.

In [ ]:
# pip install openai
from openai import OpenAI

SERVERS = {
    "llama.cpp": ("http://localhost:8080/v1",  "qwen-0.5b"),
    "ollama":    ("http://localhost:11434/v1", "qwen2.5:0.5b"),
    # "vllm":    ("http://localhost:8000/v1",  "Qwen/Qwen2.5-1.5B-Instruct"),
}

for name, (url, model) in SERVERS.items():
    client = OpenAI(base_url=url, api_key="not-needed")
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Name three Apache projects for streaming. Comma separated."}],
        temperature=0, max_tokens=40)
    print(f"{name:10s} -> {resp.choices[0].message.content!r}  ({resp.usage.completion_tokens} tokens)")

### Streaming and TTFT

In [ ]:
import time

client = OpenAI(base_url="http://localhost:11434/v1", api_key="none")
t0 = time.perf_counter(); first = None; n = 0
stream = client.chat.completions.create(
    model="qwen2.5:0.5b", stream=True, max_tokens=128,
    messages=[{"role": "user", "content": "Write a haiku about Apache Spark."}])
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    if delta and first is None:
        first = time.perf_counter() - t0
    n += 1
    print(delta, end="", flush=True)
total = time.perf_counter() - t0
print(f"\n\nTTFT {first*1000:.0f} ms · ~{n/(total-first):.1f} chunks/s (≈ tokens/s)")

## Mini-lab: latency vs throughput

`inference/bench.py` fires N concurrent requests and reports TTFT, per-request tok/s and total throughput.

```bash
python bench.py --base-url http://localhost:8080/v1  --model qwen-0.5b    --concurrency 1 4 8
python bench.py --base-url http://localhost:11434/v1 --model qwen2.5:0.5b --concurrency 1 4 8
```

::: {.fragment}
| concurrency | TTFT p50 | tok/s per request | total tok/s |
|---|---|---|---|
| 1 | 19 ms | 65 | 64 |
| 4 | 52 ms | 33 | 123 |
| 8 | | | |

Reference run: llama.cpp, Qwen2.5-0.5B Q4_K_M, 4 CPU threads, `--parallel 4`. Fill the table on your laptop. What happens beyond `--parallel` / `OLLAMA_NUM_PARALLEL`? (*queueing*: TTFT jumps)
:::

# Spark as a client{background-color="black"}

*Enriching a DataFrame with a served LLM*

## Architecture

```{mermaid}
flowchart LR
  K[(Kafka / Parquet / Iceberg)] --> S
  subgraph S[Spark job]
    P1[partition 1<br/>mapInPandas<br/>4 threads]
    P2[partition 2<br/>mapInPandas<br/>4 threads]
    P3[partition N ...]
  end
  P1 -- HTTP --> L[LLM server<br/>Ollama / vLLM<br/>max concurrent = slots]
  P2 -- HTTP --> L
  P3 -- HTTP --> L
  S --> O[(Iceberg / Elasticsearch<br/>enriched rows)]
```

::: {.fragment}
- **Concurrency budget**: `partitions running at once × threads per task` ≈ server slots (`--parallel`, `OLLAMA_NUM_PARALLEL`, `--max-num-seqs`)
- Too little → the GPU idles; too much → queueing, timeouts, retries
- **Never** a row-at-a-time UDF with a blocking call: one task = one request in flight
:::

## The pattern: `mapInPandas` + bounded thread pool

`inference/spark_enrich.py`

In [ ]:
import json, hashlib
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from openai import OpenAI
from pyspark.sql import SparkSession

BASE_URL, MODEL = "http://ollama:11434/v1", "qwen2.5:0.5b"
THREADS = 4
SYSTEM = ("Classify the tweet. Reply ONLY with JSON: "
          '{"sentiment": "positive|negative|neutral", "topic": "<max 3 words>"}')

def enrich(batches):
    client = OpenAI(base_url=BASE_URL, api_key="none", timeout=60, max_retries=3)  # once per task

    def ask(text):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=0, seed=42, max_tokens=40,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": text[:2000]}])
            out = json.loads(r.choices[0].message.content)
            return out.get("sentiment"), out.get("topic"), None
        except Exception as e:                       # one bad row must not kill the job
            return None, None, type(e).__name__

    with ThreadPoolExecutor(THREADS) as pool:
        for pdf in batches:
            res = list(pool.map(ask, pdf["text"]))
            pdf["sentiment"] = [r[0] for r in res]
            pdf["topic"] = [r[1] for r in res]
            pdf["llm_error"] = [r[2] for r in res]
            pdf["model"] = MODEL                     # lineage: which model produced the column
            pdf["prompt_sha"] = hashlib.sha256(SYSTEM.encode()).hexdigest()[:12]
            yield pdf

In [ ]:
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("LLM enrichment")
         .config("spark.sql.execution.arrow.maxRecordsPerBatch", "64")   # small batches = steady flow
         .getOrCreate())

schema = "target short, id long, ts string, flag string, user string, text string"
tweets = (spark.read.csv("/data/training.1600000.processed.noemoticon.csv.gz",
                         schema=schema, encoding="ISO-8859-1")
          .sample(fraction=0.001, seed=42)          # ~1,600 rows
          .orderBy(F.rand(42))                      # the file is sorted by label!
          .select("id", "text").limit(200)
          .repartition(2))                     # 2 tasks × 4 threads = 8 requests in flight

out_schema = ("id long, text string, sentiment string, topic string, "
              "llm_error string, model string, prompt_sha string")
enriched = tweets.mapInPandas(enrich, schema=out_schema)
enriched.write.mode("overwrite").parquet("/data/tweets_llm")
spark.read.parquet("/data/tweets_llm").groupBy("sentiment").count().show()

## Production checklist for LLM enrichment

| Concern | What to do |
|---|---|
| **Throughput** | tune partitions × threads to the server slots; prefer vLLM on GPU for big jobs |
| **Failures** | timeouts, retries with backoff, error column instead of exceptions; Spark task retries re-send requests → keep calls **idempotent** |
| **Cost of re-runs** | cache results by `hash(model, prompt, text)`; write once to Iceberg, do not recompute |
| **Determinism** | `temperature=0`, `seed`, pinned model tag/file hash |
| **Parsing** | JSON schema / structured output, validate, count invalid rows |
| **Lineage** | store `model`, `prompt_sha`, timestamp with every enriched row (→ OpenMetadata, lesson 24) |
| **Streaming** | latency per micro-batch must stay below the trigger interval; watch consumer lag |
| **Security** | the text you send is **untrusted input** (prompt injection); never let the model trigger actions directly; API key and network isolation for the server |
| **Evaluation** | compare with a labelled sample (e.g. Sentiment140 labels) before trusting the column |

## Running the whole thing

```bash
cd tap2027/inference
docker compose up -d ollama llamacpp
docker compose exec ollama ollama pull qwen2.5:0.5b
docker compose run --rm spark      # spark-submit spark_enrich.py against Ollama
```

::: {.fragment}
Switch to llama.cpp by changing two env vars in Compose:
```yaml
      LLM_BASE_URL: http://llamacpp:8080/v1
      LLM_MODEL: qwen-0.5b
```
And to vLLM on a GPU server: `http://vllm:8000/v1`. **Spark code unchanged.**
:::

## Course project: the Telegram assistant

```{mermaid}
flowchart LR
  T[Telegram bot] --> K[(Kafka)]
  K --> SP[Spark<br/>enrichment<br/>language, topic, PII flag]
  SP --> I[(Iceberg)]
  SP --> E[(Elasticsearch<br/>embeddings)]
  E --> R[RAG answer]
  R --> LLM[Ollama in dev<br/>vLLM in prod]
  LLM --> T
```

- **Batch/stream enrichment** (today): classify questions, detect language, flag personal data — via `mapInPandas`
- **Embeddings** for course documents: `/v1/embeddings` (`nomic-embed-text` on Ollama, or an ONNX model in Spark NLP!)
- **Answer generation** with retrieved, cited sources → next lesson: **Elasticsearch, vector search and RAG**

## Exercise

1. Run `bench.py` against llama.cpp and Ollama with concurrency 1/4/8 and fill the table; explain the results with the bandwidth rule
2. Change `spark_enrich.py` to extract `{"language": ..., "is_question": ...}` from Italian tweets (`training_set_sentipolc16.csv`) with a 1.5-3B model; measure invalid JSON rate and accuracy on 200 labelled rows
3. *(GPU owners)* run the same job against vLLM and compare total throughput

# Biblio

* https://github.com/ggml-org/llama.cpp — [server docs](https://github.com/ggml-org/llama.cpp/tree/master/tools/server)
* https://huggingface.co/docs/hub/gguf and https://huggingface.co/docs/hub/ollama
* https://github.com/ollama/ollama/blob/main/docs/api.md — https://ollama.com/blog/structured-outputs
* https://docs.vllm.ai/ — Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention*, SOSP 2023
* https://platform.openai.com/docs/api-reference/chat
* https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.mapInPandas.html
* https://huggingface.co/blog/introduction-to-ggml
* https://www.baseten.co/blog/llm-transformer-inference-guide/